# Performance Profile: Explanation Drift Pipeline

Measure where time is spent in a single-seed experiment run to identify
remaining bottlenecks after the metrics-selection optimization.

In [ ]:
import time
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from expl_drift import DriftDetector, DriftMonitor, AlertLevel
from expl_drift.drift.metrics import MONITORED_METRICS, compute_all_metrics
from expl_drift_experiments import (
    load_dataset, partition_into_windows, get_baseline_window,
    train_xgboost, predict_batch, evaluate_window,
    explain_shap,
)

N_WINDOWS = 20
SEED = 0
print('Imports OK')

## 1. Data loading & model training

In [ ]:
t0 = time.perf_counter()
X, y = load_dataset()
windows = partition_into_windows(X, y, n_windows=N_WINDOWS, seed=SEED)
X_base, y_base = get_baseline_window(windows)
t_data = time.perf_counter() - t0

t0 = time.perf_counter()
xgb_model, _ = train_xgboost(X_base, y_base, seed=SEED)
t_train = time.perf_counter() - t0

print(f'Data loading + windowing: {t_data:.3f}s')
print(f'XGBoost training:         {t_train:.3f}s')
print(f'Window size:              {len(X_base)} samples x {X_base.shape[1]} features')

## 2. SHAP computation (the expected bottleneck)

In [ ]:
# Baseline SHAP
t0 = time.perf_counter()
shap_baseline = explain_shap(xgb_model, X_base, X_base, model_type='xgboost')
t_shap_baseline = time.perf_counter() - t0
print(f'Baseline SHAP ({len(X_base)} samples): {t_shap_baseline:.3f}s')

# Time SHAP for one window (should be faster due to TreeExplainer caching)
X_w1, _ = windows[1]
t0 = time.perf_counter()
shap_w1 = explain_shap(xgb_model, X_w1, X_base, model_type='xgboost')
t_shap_w1 = time.perf_counter() - t0
print(f'Window SHAP ({len(X_w1)} samples, cached explainer): {t_shap_w1:.3f}s')

# Time all 19 remaining windows
t0 = time.perf_counter()
all_shap = [shap_baseline]
for wid in range(1, N_WINDOWS):
    X_w, _ = windows[wid]
    all_shap.append(explain_shap(xgb_model, X_w, X_base, model_type='xgboost'))
t_shap_all = time.perf_counter() - t0
print(f'All 19 window SHAP calls:  {t_shap_all:.3f}s ({t_shap_all/19:.3f}s each)')

## 3. Metric computation: monitored-only vs all

In [ ]:
# Monitored-only (fast path)
t0 = time.perf_counter()
for s in all_shap[1:]:
    compute_all_metrics(shap_baseline, s, metrics=MONITORED_METRICS)
t_fast = time.perf_counter() - t0

# All metrics (old path)
t0 = time.perf_counter()
for s in all_shap[1:]:
    compute_all_metrics(shap_baseline, s)
t_all = time.perf_counter() - t0

print(f'Monitored-only (19 windows): {t_fast:.3f}s ({t_fast/19*1000:.1f}ms each)')
print(f'All metrics (19 windows):    {t_all:.3f}s ({t_all/19*1000:.1f}ms each)')
print(f'Speedup: {t_all/t_fast:.1f}x')

## 4. Full DriftMonitor pipeline (calibration + 19 evaluations)

In [ ]:
detector = DriftDetector(shap_baseline)
calibration_shap = all_shap[1:5]

t0 = time.perf_counter()
monitor = DriftMonitor(detector, calibration_shap, warning_std=2.5, critical_std=3.5)
t_calibrate = time.perf_counter() - t0

t0 = time.perf_counter()
for s in all_shap[1:]:
    monitor.evaluate(s)
t_evaluate = time.perf_counter() - t0

print(f'Monitor calibration (4 windows): {t_calibrate:.3f}s')
print(f'Monitor evaluation (19 windows): {t_evaluate:.3f}s ({t_evaluate/19*1000:.1f}ms each)')

## 5. Per-component breakdown for a single seed

In [ ]:
total_shap = t_shap_baseline + t_shap_all
total_monitor = t_calibrate + t_evaluate
total = t_data + t_train + total_shap + total_monitor

print('=== Single Seed Time Budget ===')
print(f'  Data + windowing:  {t_data:6.3f}s  ({t_data/total*100:4.1f}%)')
print(f'  Model training:    {t_train:6.3f}s  ({t_train/total*100:4.1f}%)')
print(f'  SHAP computation:  {total_shap:6.3f}s  ({total_shap/total*100:4.1f}%)')
print(f'  Monitor (cal+eval):{total_monitor:6.3f}s  ({total_monitor/total*100:4.1f}%)')
print(f'  ──────────────────────────────')
print(f'  TOTAL:             {total:6.3f}s')
print()
print(f'Projected 25 seeds:  {total * 25:.0f}s ({total * 25 / 60:.1f} min)')
print(f'Projected 5 experiments x 25 seeds: {total * 125:.0f}s ({total * 125 / 60:.1f} min)')

## 6. Individual metric timings

In [ ]:
from expl_drift.drift.metrics import (
    compute_cosine_attribution_drift,
    compute_jsd, compute_max_jsd,
    compute_wasserstein_drift, compute_max_wasserstein_drift,
    compute_energy_distance,
    compute_ks_statistic,
    compute_classifier_drift,
)

metrics_fns = [
    ('cosine_drift', lambda: compute_cosine_attribution_drift(shap_baseline, all_shap[5])),
    ('jsd', lambda: compute_jsd(shap_baseline, all_shap[5])),
    ('max_jsd', lambda: compute_max_jsd(shap_baseline, all_shap[5])),
    ('wasserstein', lambda: compute_wasserstein_drift(shap_baseline, all_shap[5])),
    ('max_wasserstein', lambda: compute_max_wasserstein_drift(shap_baseline, all_shap[5])),
    ('energy_distance', lambda: compute_energy_distance(shap_baseline, all_shap[5])),
    ('ks_statistic', lambda: compute_ks_statistic(shap_baseline, all_shap[5])),
    ('classifier_drift', lambda: compute_classifier_drift(shap_baseline, all_shap[5])),
]

N_ITER = 20
print(f'Per-metric timing ({N_ITER} iterations each):')
print(f'{"Metric":<25} {"Mean (ms)":>10} {"Used in alerting":>18}')
print('-' * 55)
for name, fn in metrics_fns:
    t0 = time.perf_counter()
    for _ in range(N_ITER):
        fn()
    elapsed = (time.perf_counter() - t0) / N_ITER * 1000
    used = 'YES' if name in ('cosine_drift', 'max_jsd', 'max_wasserstein') else 'no'
    print(f'{name:<25} {elapsed:>9.2f}  {used:>18}')